# Lab 04: The Vision Transformer front end and forward pass

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hokwangchoi/hc-model-study/blob/master/labs/notebooks/lab04_vit.ipynb) · lab page: [`labs/lab04.html`](../lab04.html) · file: [`labs/lab04_vit.py`](../lab04_vit.py)

**Time:** 45-60 min

## Goal

Turn an image into a sequence of tokens and classify it with a transformer
encoder. You implement patchify twice (pure reshapes, and a strided Conv2d)
and show they are the same linear map, then the CLS token, the position
embeddings, the forward pass, and a shape tracker. The transformer blocks
themselves are torch's (you built one in lab 02).

## The maths, step by step

```text
B batch, C channels, H x W image, P patch size (argument p in the code),
d width, K classes (as in the ViT guide). N counts patches; N+1 tokens with CLS.

  1. Patchify      [B, C, H, W] -> [B, N, C*P*P],   N = (H/P) * (W/P)
                   cut the image into non-overlapping P x P tiles, row by
                   row; flatten each tile in (channel, row, column) order.
  2. Patch embed   tokens = patches @ W_e^T + b       W_e: [d, C*P*P]  -> [B, N, d]
                   Identical to Conv2d(C, d, kernel_size=P, stride=P): the
                   kernel visits each tile once and takes a dot product with
                   it. conv.weight [d, C, P, P] reshaped to [d, C*P*P] is W_e.
  3. CLS token     prepend one learned vector:  [B, N, d] -> [B, N+1, d]
                   It owns no patch; through attention it gathers a summary
                   of the whole image, and the classifier reads only it.
  4. Positions     x = x + pos_embed                  pos_embed: [1, N+1, d]
                   Attention is permutation-equivariant: without this,
                   shuffling the patches would not change the CLS output.
  5. Encoder       L pre-norm blocks, bidirectional (no causal mask)  [B, N+1, d]
  6. Head          LayerNorm, take token 0 (CLS), Linear(d, K)        [B, K]

  ViT-B/16 at 224 x 224: N = 14 * 14 = 196 patches, 197 tokens, d = 768.
```

## Read alongside

- [ViT guide](../../vit/vit_guide.html): section 2.1 High-Level Inference Pipeline (shape flow), 2.2 Patch Embedding, 2.3 Position Embeddings, 2.5 Full Model: Where the Parameters and Compute Go, and #python-code (condensed reference implementation)

## How to work

1. Run the setup cell and the imports cell.
2. Implement the functions in order, replacing `raise NotImplementedError  # TODO`. The hints above each cell open one at a time; try Hint 1 first.
3. Run each cell after editing it, then run the check cell at the end: `check.run_lab(4, namespace=globals())` checks the functions defined in this notebook (TODO = not started, PASS = matches the reference, FAIL = read the message).

Your work stays in this notebook; it does not change `labs/lab04_vit.py` or the lab page. The reference solution is `labs/solutions/lab04_vit.py`.

In [ ]:
# Setup: make the checker (labs/check.py) importable. Run this cell first.
# In a clone of the repository (Jupyter, VS Code) it uses labs/check.py;
# elsewhere (e.g. Google Colab) it downloads check.py, which needs only torch.
import pathlib
import sys
import urllib.request


def _find_labs_dir():
    here = pathlib.Path.cwd().resolve()
    for d in [here, *here.parents]:
        for cand in (d, d / "labs"):
            if (cand / "check.py").is_file() and (cand / "solutions").is_dir():
                return cand
    return None


LABS_DIR = _find_labs_dir()
if LABS_DIR is None:
    LABS_DIR = pathlib.Path("hc_model_study_labs").resolve()
    LABS_DIR.mkdir(exist_ok=True)
    if not (LABS_DIR / "check.py").is_file():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/hokwangchoi/hc-model-study/master/labs/check.py", LABS_DIR / "check.py")
sys.path.insert(0, str(LABS_DIR))
import check  # noqa: E402

import torch  # noqa: E402
print("checker:", LABS_DIR / "check.py", "| torch", torch.__version__)

### Imports

In [ ]:
from dataclasses import dataclass

import torch
import torch.nn as nn

## Step 1: patchify with reshapes (and its inverse)

**Hints for `patchify`** (open one at a time):

<details><summary>Hint 1</summary>

```text
split H into (H/p, p) and W into (W/p, p) with one .reshape:
[B, C, H/p, p, W/p, p].
```

</details>

<details><summary>Hint 2</summary>

```text
move the two grid dims to the front of each sample and the three
per-patch dims (C, p, p) to the back: permute(0, 2, 4, 1, 3, 5).
```

</details>

<details><summary>Hint 3</summary>

```text
images.reshape(B, C, H // p, p, W // p, p).permute(0, 2, 4, 1, 3, 5)
    .reshape(B, (H // p) * (W // p), C * p * p)
```

</details>

In [ ]:
def patchify(images: torch.Tensor, p: int) -> torch.Tensor:
    """[B, C, H, W] -> [B, N, C*p*p].

    Patch n = (row r, column c) of the patch grid, n = r * (W/p) + c.
    Its vector is images[b, :, r*p:(r+1)*p, c*p:(c+1)*p].reshape(-1),
    i.e. ordered (channel, row inside the patch, column inside the patch).
    """
    raise NotImplementedError  # TODO

**Hints for `unpatchify`** (open one at a time):

<details><summary>Hint 1</summary>

```text
run patchify's steps backwards: reshape, inverse permute, reshape.
```

</details>

<details><summary>Hint 2</summary>

```text
[B, gh, gw, C, p, p] -> permute(0, 3, 1, 4, 2, 5) -> [B, C, gh, p, gw, p]
```

</details>

In [ ]:
def unpatchify(patches: torch.Tensor, C: int, H: int, W: int, p: int) -> torch.Tensor:
    """Inverse of patchify: [B, N, C*p*p] -> [B, C, H, W].

    (DiT, the diffusion transformer, ends with exactly this step.)
    """
    raise NotImplementedError  # TODO

## Step 2: patch embedding, two ways

**Hints for `PatchEmbed.forward_conv`** (open one at a time):

<details><summary>Hint 1</summary>

```text
self.proj(x) is [B, d, H/p, W/p]: a d-channel feature map, one pixel per patch.
```

</details>

<details><summary>Hint 2</summary>

```text
flatten the two grid dims, then put tokens before channels.
```

</details>

<details><summary>Hint 3</summary>

```text
return self.proj(x).flatten(2).transpose(1, 2)
```

</details>

**Hints for `PatchEmbed.forward_linear`** (open one at a time):

<details><summary>Hint 1</summary>

```text
the conv weight [d, C, p, p] flattened to [d, C*p*p] is a Linear weight
whose input order (C, p, p) matches your patchify.
```

</details>

<details><summary>Hint 2</summary>

```text
w = self.proj.weight.reshape(self.proj.out_channels, -1)
```

</details>

<details><summary>Hint 3</summary>

```text
return patchify(x, self.patch_size) @ w.T + self.proj.bias
```

</details>

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size: int = 32, patch_size: int = 8, in_chans: int = 3, dim: int = 48):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, dim, kernel_size=patch_size, stride=patch_size)

    def forward_conv(self, x: torch.Tensor) -> torch.Tensor:
        """[B, C, H, W] -> [B, N, d] using the strided convolution self.proj."""
        raise NotImplementedError  # TODO

    def forward_linear(self, x: torch.Tensor) -> torch.Tensor:
        """[B, C, H, W] -> [B, N, d] using patchify() and a matmul with the SAME weights."""
        raise NotImplementedError  # TODO

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.forward_conv(x)

## Step 3: the ViT forward pass (CLS token, position embeddings, encoder, head)

Given: `class ViTConfig` is already written. Run the cell.

In [ ]:
@dataclass
class ViTConfig:
    img_size: int = 32
    patch_size: int = 8
    in_chans: int = 3
    dim: int = 48
    depth: int = 2
    n_heads: int = 4
    n_classes: int = 10

**Hints for `ViT.forward`** (open one at a time):

<details><summary>Hint 1</summary>

```text
order: patch_embed -> prepend CLS -> add pos_embed -> blocks -> norm
-> take token 0 -> head. Each block is called as x = block(x).
```

</details>

<details><summary>Hint 2</summary>

```text
the CLS parameter is [1, 1, d]; give every sample its copy with
self.cls_token.expand(B, -1, -1), then torch.cat([cls, x], dim=1).
CLS goes FIRST, at index 0, which is where the head reads it.
```

</details>

<details><summary>Hint 3</summary>

```text
to record a shape: if trace is not None: trace["tokens"] = tuple(x.shape)
```

</details>

In [ ]:
class ViT(nn.Module):
    def __init__(self, cfg: ViTConfig):
        super().__init__()
        self.cfg = cfg
        self.patch_embed = PatchEmbed(cfg.img_size, cfg.patch_size, cfg.in_chans, cfg.dim)
        n = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(0.02 * torch.randn(1, 1, cfg.dim))
        self.pos_embed = nn.Parameter(0.02 * torch.randn(1, n + 1, cfg.dim))
        # Pre-norm encoder blocks (LayerNorm -> MHA -> add; LayerNorm -> MLP -> add), as in lab 02.
        self.blocks = nn.ModuleList(
            nn.TransformerEncoderLayer(cfg.dim, cfg.n_heads, 4 * cfg.dim, dropout=0.0,
                                       activation="gelu", batch_first=True, norm_first=True)
            for _ in range(cfg.depth))
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, cfg.n_classes)

    def forward(self, images: torch.Tensor, trace: dict = None) -> torch.Tensor:
        """images [B, C, H, W] -> logits [B, n_classes].

        If `trace` is a dict, also record tuple(shape) under these keys:
            "input"         the images                       [B, C, H, W]
            "patch_tokens"  after patch embedding            [B, N, d]
            "tokens"        after CLS + position embeddings  [B, N+1, d]
            "encoded"       after the blocks and self.norm   [B, N+1, d]
            "cls"           the CLS feature                  [B, d]
            "logits"        the output                       [B, n_classes]
        """
        raise NotImplementedError  # TODO

## Step 4: predict the shapes without running anything

**Hints for `vit_shapes`** (open one at a time):

<details><summary>Hint 1</summary>

```text
N = (img_size // patch_size) ** 2; the CLS token adds one.
```

</details>

<details><summary>Hint 2</summary>

```text
return {"input": (batch, in_chans, img_size, img_size),
        "patch_tokens": (batch, N, dim), ...}  (six keys)
```

</details>

In [ ]:
def vit_shapes(batch: int, img_size: int, patch_size: int, in_chans: int, dim: int,
               n_classes: int) -> dict:
    """Return the shapes forward() records in `trace`, computed by hand (tuples of ints).

    Example: vit_shapes(1, 224, 16, 3, 768, 1000)["tokens"] == (1, 197, 768).
    """
    raise NotImplementedError  # TODO

## Check

Run this after each function you finish. Steps you have not started show TODO; finished steps show PASS. For a full traceback of a failing step: `check.run_lab(4, globals(), verbose=True)`.

In [ ]:
check.run_lab(4, namespace=globals());

## Scratch

Try your functions here: print shapes, call them on small inputs. (Functions you have not written yet raise `NotImplementedError`.)

In [ ]:
# Try the pieces here (Run the cells above first).
img = torch.randn(2, 3, 32, 32)
print(patchify(img, 8).shape)         # [2, 16, 192]
trace = {}
logits = ViT(ViTConfig())(img, trace)
trace